# The Problem Nobody Warns You About

You downloaded ERA5. It's 2.3 GB. You opened it with xarray. Your laptop froze. Sound familiar?

This tutorial teaches you WHY that happens and exactly how to extract a clean time series for any point location without loading a single unnecessary byte.

Real problem: Extract Hs, Tp, wind speed at [4°N, 108°E] (Natuna Sea, offshore platform) from 10 years of ERA5.


### SECTION 1: How Gridded Data Actually Works

#### 1.1 What is a NetCDF file, really?

In [1]:
# Conceptual diagram — show this as a markdown cell with ASCII art

# NetCDF = labeled N-dimensional array on disk
#
#          time (8760 steps)
#         /
#        /
#       /_________ lat (721 points)
#       |
#       |_________ lon (1440 points)
#
# Each variable (Hs, Tp, wind) = one cube
# You want ONE point → a line through the time axis
# Loading the whole cube for one point = criminal waste


#### 1.2 Lazy vs Eager Loading — The Core Concept

Lazy Loading: "I'll read it when I actually need it"
Eager Loading: "Load everything into RAM right now"

| Method           | Loads to RAM? | Speed to open | Speed to compute |
|------------------|---------------|---------------|------------------|
| pandas read_csv  | Eager — YES   | Fast          | Fast (small data)|
| xarray (default) | Lazy          | Instant       | Fast (on demand) |
| xarray + dask    | Lazy + chunked| Instant       | Parallelized     |
| netCDF4 raw      | Eager         | Slow          | Depends          |

Rule: For files > 500MB — ALWAYS lazy load first.


#### 1.3 Indexing Methods — sel vs isel vs interp

In [ ]:
# Three ways to select a point — teach the difference clearly

from pathlib import Path

import xarray as xr

# Pre-sliced sample shipped in repo: data/era5_sample.nc (~few MB)
for path in (Path("../data/era5_sample.nc"), Path("era5_sample.nc")):
    if path.exists():
        break
else:
    raise FileNotFoundError("Missing era5_sample.nc — clone repo or run scripts/download_era5.py")

ds = xr.open_dataset(path)  # lazy by default

# METHOD 1: Label-based selection (nearest grid point)
# Use when: you want exact lat/lon, snapping to nearest grid
ts_sel = ds["swh"].sel(latitude=4.0, longitude=108.0, method="nearest")

# METHOD 2: Index-based selection
# Use when: you know the array index position
ts_isel = ds["swh"].isel(latitude=42, longitude=864)

# METHOD 3: Interpolation (bilinear)
# Use when: your point sits between grid cells, precision matters
ts_interp = ds["swh"].interp(latitude=4.0, longitude=108.0)

# KEY INSIGHT: sel/isel = lazy (nothing loaded yet)
#              .values or .to_dataframe() = triggers actual read


#### 1.4 Memory Model — What Actually Happens

Think of xarray like a map of a library.
- Opening the dataset = reading the catalog (instant, tiny)
- .sel() = marking a book on the shelf (still lazy)
- .values = physically walking to the shelf and pulling the book

This is why xarray feels "instant to open" — 
it hasn't read your 2.3GB file yet.

### SECTION 2: Data Setup

In [ ]:
# Pre-sliced file already in repo — run this and you're ready
import urllib.request

url = "link from gdrive to store huge data, or alternative platform to be downloaded easily"
urllib.request.urlretrieve(url, "era5_sample.nc")
print("✅ Sample data ready")

### SECTION 3: The Extraction — Four Methods, One Goal

#### 3.1 Method 1 — Naive (The Wrong Way, for contrast)

In [ ]:
# THE WRONG WAY — shown intentionally
# Don't do this with large files

import xarray as xr
import time

start = time.time()
ds_eager = xr.open_dataset("era5_sample.nc", 
                            chunks=None)  # forces eager
hs_wrong = ds_eager["swh"].values  # loads EVERYTHING
point_wrong = hs_wrong[:, 42, 864]  # then slices
elapsed = time.time() - start

print(f"Loaded full array: {hs_wrong.nbytes / 1e6:.1f} MB")
print(f"Time: {elapsed:.2f}s")
print("❌ Don't do this on a 2GB file — you'd load 2GB for one point")


#### 3.2 Method 2 — xarray Lazy + .sel() (The Standard Way)

In [ ]:
start = time.time()

ds = xr.open_dataset("era5_sample.nc")  # lazy
hs_point = ds["swh"].sel(
    latitude=4.0, 
    longitude=108.0, 
    method="nearest"
).compute()  # only NOW does it read from disk

elapsed = time.time() - start
print(f"Extracted {len(hs_point)} timesteps")
print(f"Memory used: {hs_point.nbytes / 1e3:.1f} KB")
print(f"Time: {elapsed:.2f}s ✅")


#### 3.3 Method 3 — xarray + Dask (For Very Large Files)


In [ ]:
start = time.time()

ds_dask = xr.open_dataset(
    "era5_natuna_10yr.nc",   # use the big file here
    chunks={"time": 1000}    # chunk along time axis
)

hs_dask = ds_dask["swh"].sel(
    latitude=4.0,
    longitude=108.0,
    method="nearest"
).compute()  # dask executes the graph

elapsed = time.time() - start
print(f"Dask extraction time: {elapsed:.2f}s")


💡 When to use Dask:
- File > 1 GB
- Multi-year, multi-variable analysis
- Running on a machine with multiple CPU cores

For single-point extraction from a pre-sliced file,
standard xarray is usually faster (less overhead).

#### 3.4 Method 4 — Interpolation (When Precision Matters)

In [ ]:
# Bilinear interpolation — for points between grid cells
# ERA5 resolution: 0.25° × 0.25° (~28 km)
# Your platform: 4.12°N, 108.37°E — between grid points

hs_interp = ds["swh"].interp(
    latitude=4.12,
    longitude=108.37,
    method="linear"
).compute()

# Compare nearest vs interpolated
import numpy as np
diff = np.abs(hs_point.values - hs_interp.values).mean()
print(f"Mean difference (nearest vs interp): {diff:.4f} m")
print("→ For offshore engineering, use interp for site-specific reports")


### SECTION 4: Post-Processing — Clean Output

In [ ]:
import pandas as pd

# Convert to DataFrame — the universal handoff format
df = pd.DataFrame({
    "datetime": hs_point.time.values,
    "hs_m": hs_point.values,
    "lat": 4.0,
    "lon": 108.0,
    "source": "ERA5"
})

df["datetime"] = pd.to_datetime(df["datetime"])
df = df.set_index("datetime")

# Quick sanity check
print(df.describe())
print(f"\nMissing values: {df['hs_m'].isna().sum()}")

# Export
df.to_csv("natuna_hs_timeseries.csv")
print("✅ Saved: natuna_hs_timeseries.csv")


### SECTION 5: Benchmark — The Performance Comparison


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

methods = {}

# Benchmark 1: Naive (eager full load)
start = time.time()
ds_e = xr.open_dataset("era5_sample.nc", chunks=None)
_ = ds_e["swh"].values[:, 42, 864]
methods["Naive\n(Full Load)"] = time.time() - start

# Benchmark 2: xarray lazy + sel
start = time.time()
ds_l = xr.open_dataset("era5_sample.nc")
_ = ds_l["swh"].sel(latitude=4.0, longitude=108.0, 
                     method="nearest").compute()
methods["xarray\nLazy+sel"] = time.time() - start

# Benchmark 3: xarray + dask
start = time.time()
ds_d = xr.open_dataset("era5_sample.nc", chunks={"time": 500})
_ = ds_d["swh"].sel(latitude=4.0, longitude=108.0,
                     method="nearest").compute()
methods["xarray\n+Dask"] = time.time() - start

# Benchmark 4: interp
start = time.time()
ds_i = xr.open_dataset("era5_sample.nc")
_ = ds_i["swh"].interp(latitude=4.12, longitude=108.37).compute()
methods["xarray\nInterp"] = time.time() - start

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(methods.keys(), methods.values(), 
              color=["#e74c3c", "#2ecc71", "#3498db", "#9b59b6"],
              width=0.5)
ax.set_ylabel("Time (seconds)", fontsize=12)
ax.set_title("Point Extraction Performance — ERA5 Sample File\n"
             "Nusawave Labs · extract-point tutorial", fontsize=13)
for bar, val in zip(bars, methods.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.2f}s", ha="center", fontsize=11, fontweight="bold")
ax.set_ylim(0, max(methods.values()) * 1.3)
plt.tight_layout()
plt.savefig("benchmark_results.png", dpi=150)
plt.show()


## What This Chart Tells You

| Method | ~Time | Use When |
|---|---|---|
| Naive full load | Slowest | Never, for large files |
| xarray lazy+sel | Fast | Default choice, always |
| xarray+Dask | Moderate overhead | Files > 1GB, multi-core |
| xarray interp | ~Same as sel | Site-specific precision needed |

🔑 Key insight: The "lazy" approach isn't just faster — 
it scales. On a 40GB multi-year file, naive loading 
crashes your machine. Lazy+sel still works in seconds.


### SECTION 6: What's Next

##### You Just Learned:
✅ Why NetCDF files crash laptops (eager loading)
✅ How lazy loading works (xarray's superpower)
✅ Three extraction methods + when to use each
✅ How to export a clean time series CSV

##### Next Tutorial:
📍 "Your First Ocean Map: Plot SST and Currents 
    in 20 Lines of Python"
→ We'll use the CSV you just created as input.

##### What Real Engineers Do Next With This Data:
- Wave statistics & exceedance curves (M6)
- Extreme value analysis for design criteria (M8)
- Joint extremes for offshore structure design (M9)

Follow Nusawave on [LinkedIn](https://www.linkedin.com/company/109905023)

Star this repo on [GitHub](github.com/nusawavelab/extract-point)
